# M1 — KV-Injection Knowledge Displacement

**Goal:** Validate that workspace KV injection can displace in-weights knowledge at the MiniCPM5-1B scale.

**G6 criterion:** `mount_accuracy / weights_accuracy ≥ 0.6`

**Architecture change from M0.5:** Instead of prepending workspace text (which requires in-context learning), we encode workspace facts into per-layer K/V tensors using the model's own projections, then inject them directly into each attention layer. All three branches attend to workspace KVs naturally.

**Runtime:** Set to **A100 GPU** (Runtime → Change runtime type → A100).

In [ ]:
# 1. Setup
import os, subprocess, sys, json
from pathlib import Path

REPO_URL = os.environ.get('LOCALSPARSE_REPO', 'https://github.com/kaaninel/localsparse')
REPO_DIR = Path('/content/localsparse')
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -e . transformers==5.9.0 accelerate
sys.path.insert(0, str(REPO_DIR))

import torch
print('cuda?', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total --format=csv,noheader

In [ ]:
# 2. Config
from pathlib import Path
from datetime import datetime

OUT = Path('/content/runs/m1')
OUT.mkdir(parents=True, exist_ok=True)

# === CHOOSE MODEL ===
# 'minicpm' = MiniCPM5-1B (primary, ~2GB download, needs A100)
# 'veyra3'  = Veyra3-5M   (fast sanity check, ~20MB download)
MODEL_TARGET = 'minicpm'

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'bfloat16'

# G6 training params
N_FACTS = 200          # number of held-out facts for mount-path
EPOCHS  = 60           # training epochs for weights-path
BATCH   = 16           # batch size
SEQ_LEN = 512
LR      = 1e-3
BANK_MAX_LEN = 1024    # max tokens for workspace KV encoding
G6_THRESHOLD = 0.6

print(f'OUT={OUT} MODEL={MODEL_TARGET} DEV={DEV}')

In [ ]:
# 3. Run M1 KV-injection pipeline
!python scripts/run_m1_kv_injection.py \
    --model {MODEL_TARGET} \
    --device {DEV} \
    --dtype {DTYPE} \
    --n_facts {N_FACTS} \
    --epochs {EPOCHS} \
    --batch_size {BATCH} \
    --seq_len {SEQ_LEN} \
    --lr {LR} \
    --bank_max_length {BANK_MAX_LEN} \
    --g6_threshold {G6_THRESHOLD} \
    --run_dir {OUT}/run_{MODEL_TARGET}

In [ ]:
# 4. Show results
import json
summary_path = OUT / f'run_{MODEL_TARGET}' / 'summary.json'
if summary_path.exists():
    result = json.loads(summary_path.read_text())
    print(json.dumps(result, indent=2))

    ratio = result['value']
    status = result['status']
    print(f'\n=== G6 HEADLINE ===')
    print(f"  weights-path accuracy: {result['weights_accuracy']:.3f}")
    print(f"  mount-path accuracy:   {result['mount_accuracy']:.3f}")
    print(f"  control (no mount):    {result['control_accuracy']:.3f}")
    print(f"  ratio:                 {ratio:.3f}")
    print(f"  threshold:             {result['threshold']}")
    if status == 'pass':
        print('\n✅ G6 PASS — KV injection displaces weights-knowledge!')
        print('Next step: run full M1 training pipeline (M1-M9 milestones)')
    else:
        print(f'\n❌ G6 FAIL (ratio={ratio:.3f})')
        print('Diagnosis:')
        print('  1. Is mount_accuracy > control_accuracy? (injection must add signal)')
        print('  2. Try --bank_max_length 2048 --epochs 120')
        print('  3. Check surgery report (all layers replaced?)')
else:
    print('No summary.json found — check cell 3 output for errors')

In [ ]:
# 5. [Optional] Diagnostic: verify injection changes logits
# This cell manually checks that the KV bank injection is actually
# affecting model output, independently of accuracy.

import torch, sys
from pathlib import Path
sys.path.insert(0, '/content/localsparse')

from localsparse.workspace.kv_bank import WorkspaceKVBank
from localsparse.training.factoid_world import build_world, build_qa_pairs

# Reload model for diagnostics
if MODEL_TARGET == 'minicpm':
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from localsparse.config import LocalSparseConfig, ModelDims, AttentionConfig
    from localsparse.model.minicpm_adapter import surgery_minicpm
    diag_model = AutoModelForCausalLM.from_pretrained(
        'openbmb/MiniCPM5-1B', torch_dtype=torch.bfloat16, trust_remote_code=True).to(DEV)
    diag_tok = AutoTokenizer.from_pretrained('openbmb/MiniCPM5-1B', trust_remote_code=True)
    diag_cfg = LocalSparseConfig(
        model=ModelDims(
            vocab_size=diag_model.config.vocab_size,
            hidden_size=diag_model.config.hidden_size,
            num_layers=diag_model.config.num_hidden_layers,
            num_q_heads=diag_model.config.num_attention_heads,
            num_kv_heads=getattr(diag_model.config, 'num_key_value_heads', diag_model.config.num_attention_heads),
            head_dim=getattr(diag_model.config, 'head_dim', diag_model.config.hidden_size // diag_model.config.num_attention_heads),
        ),
        attention=AttentionConfig(sliding_window=4096, compressed_block=64,
                             super_block=4096, selected_top_k=16, indexer_dim=64),
    )
    surgery_minicpm(diag_model, diag_cfg)
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from localsparse.model.veyra_adapter import surgery_veyra3, VeyraAdapterConfig
    diag_model = AutoModelForCausalLM.from_pretrained(
        'veyra-ai/veyra3-5m-base', torch_dtype=torch.bfloat16, trust_remote_code=True).to(DEV)
    diag_tok = AutoTokenizer.from_pretrained('veyra-ai/veyra3-5m-base', trust_remote_code=True)
    surgery_veyra3(diag_model, VeyraAdapterConfig())

bank_diag = WorkspaceKVBank()
bank_diag.encode(diag_model, 'The capital of France is Paris.', diag_tok, DEV, max_length=64)

test_ids = diag_tok('What is the capital of France?', return_tensors='pt')['input_ids'].to(DEV)

diag_model.eval()
with torch.no_grad():
    base_logits = diag_model(input_ids=test_ids).logits[0, -1]

with bank_diag.inject(diag_model):
    with torch.no_grad():
        mounted_logits = diag_model(input_ids=test_ids).logits[0, -1]

diff = (base_logits - mounted_logits).abs().max().item()
print(f'Max logit diff (base vs mounted): {diff:.6f}')
print('✅ Injection is working' if diff > 1e-4 else '❌ No difference — injection may be broken')

# Show top predictions with/without mount
top_base = base_logits.topk(5)
top_mount = mounted_logits.topk(5)
print('\nTop-5 without mount:', [diag_tok.decode([t]) for t in top_base.indices.tolist()])
print('Top-5 with mount:   ', [diag_tok.decode([t]) for t in top_mount.indices.tolist()])